# Fold-In — Personal User Vector
Hold the SVD item vectors fixed and solve for a personal user vector via least-squares using the 35 MovieLens-matched ratings.

SVD predicts a rating as: `r = global_mean + b_u + b_i + u · q_i`  
Known: `global_mean`, `b_i` (item biases), `q_i` (item vectors), `r` (personal ratings)  
Unknown: `u` (user vector, length k=20), `b_u` (user bias, scalar)

Rearranging: `r - global_mean - b_i ≈ b_u + u · q_i`  
This is a linear system — stack all 35 equations and solve with least-squares.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

data_path = Path('../data')

## Load item vectors and personal ratings

In [ ]:
d = np.load(data_path / 'svd_item_vectors.npz')
item_vectors = d['item_vectors']          # (n_items, k)
item_biases  = d['item_biases']           # (n_items,)
movieids     = d['movieids']              # (n_items,)
global_mean  = d['global_mean'][0]        # scalar

movieid_to_idx = {mid: i for i, mid in enumerate(movieids)}

print(f"Item matrix: {item_vectors.shape}")
print(f"Global mean: {global_mean:.4f}")

matched = pd.read_csv(data_path / 'movielens_matched.csv')
print(f"Personal matched ratings: {len(matched)}")
matched[['letterboxd_title', 'movieId', 'rating']].head()

## Build the least-squares system

In [ ]:
rows = []
for _, row in matched.iterrows():
    idx = movieid_to_idx.get(int(row['movieId']))
    if idx is None:
        continue
    qi    = item_vectors[idx]
    bi    = item_biases[idx]
    r     = row['rating']
    title = row['letterboxd_title']
    rows.append((qi, bi, r, title))

print(f"Ratings usable for fold-in: {len(rows)}")

# Target: r - global_mean - b_i
# Features: [1, q_i]  (1 for the user bias term)
A = np.column_stack([np.ones(len(rows)), np.array([row[0] for row in rows])])  # (n, k+1)
b = np.array([row[2] - global_mean - row[1] for row in rows])                  # (n,)

print(f"A shape: {A.shape}  (equations × unknowns)")

## Solve for user vector

In [ ]:
solution, residuals, rank, sv = np.linalg.lstsq(A, b, rcond=None)

user_bias   = solution[0]
user_vector = solution[1:]

print(f"User bias (b_u): {user_bias:.4f}")
print(f"User vector shape: {user_vector.shape}")
print(f"User vector norm: {np.linalg.norm(user_vector):.4f}")

## Sanity check — predict back the training ratings

In [ ]:
preds, actuals, titles = [], [], []
for (qi, bi, r, title) in rows:
    pred = global_mean + user_bias + bi + user_vector @ qi
    preds.append(pred)
    actuals.append(r)
    titles.append(title)

results = pd.DataFrame({'title': titles, 'actual': actuals, 'predicted': preds})
results['error'] = results['predicted'] - results['actual']
rmse = np.sqrt((results['error'] ** 2).mean())
print(f"Train RMSE (fold-in): {rmse:.4f}")
results.sort_values('error', key=abs, ascending=False)

## Alternative: Ridge regression

Plain least-squares ignores the regularization used during SVD training (`reg=0.1`).
Ridge adds an L2 penalty `λ||u||²` that keeps the user vector in the same scale as the item vectors.

Solves: `(AᵀA + λI)x = Aᵀb`  via [`sklearn.linear_model.Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html).
`λ = 0.1` matches the training regularization strength.

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=0.1, fit_intercept=True)
ridge.fit(np.array([row[0] for row in rows]), b)

user_bias_ridge   = ridge.intercept_
user_vector_ridge = ridge.coef_

print(f"Ridge  — user bias: {user_bias_ridge:.4f}  |  vector norm: {np.linalg.norm(user_vector_ridge):.4f}")
print(f"lstsq  — user bias: {user_bias:.4f}  |  vector norm: {np.linalg.norm(user_vector):.4f}")

# Fold-in RMSE for Ridge
preds_ridge = [
    global_mean + user_bias_ridge + row[1] + user_vector_ridge @ row[0]
    for row in rows
]
rmse_ridge = np.sqrt(np.mean((np.array(preds_ridge) - np.array(actuals)) ** 2))
print(f"\nFold-in RMSE — lstsq: {rmse:.4f}  |  Ridge: {rmse_ridge:.4f}")

## Tune alpha with Leave-One-Out Cross-Validation

`alpha=0.1` matched SVD training regularization, but the optimal fold-in value may differ — especially with only 35 ratings and 21 unknowns.

[`RidgeCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html) efficiently evaluates all candidate alphas in one pass using LOOCV (`cv=len(rows)`), then re-fits on the full dataset with the best alpha.

In [ ]:
from sklearn.linear_model import RidgeCV

Q        = np.array([row[0] for row in rows])                            # (n, k) item vectors
b_target = np.array([row[2] - global_mean - row[1] for row in rows])    # adjusted targets

alphas   = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

ridge_cv = RidgeCV(alphas=alphas, fit_intercept=True,
                   scoring='neg_mean_squared_error',
                   cv=len(rows))   # cv=n → true LOOCV
ridge_cv.fit(Q, b_target)

user_bias_cv   = ridge_cv.intercept_
user_vector_cv = ridge_cv.coef_

print(f"Best alpha (LOOCV): {ridge_cv.alpha_}")
print(f"User bias  (b_u):   {user_bias_cv:.4f}")
print(f"User vector norm:   {np.linalg.norm(user_vector_cv):.4f}")

## Sanity checks

1. **Vector norm** — compare user vector norm to the average item vector norm. If the user norm is much larger, alpha is still too low.
2. **Train RMSE** — should be similar to the fixed-alpha Ridge above; a big jump means alpha is too high (underfitting).
3. **Top predictions** — gut-check that the highest-predicted movies match your actual taste.

In [ ]:
# 1. Norm comparison
avg_item_norm = np.linalg.norm(item_vectors, axis=1).mean()
print(f"Avg item vector norm : {avg_item_norm:.4f}")
print(f"User vector norm     : {np.linalg.norm(user_vector_cv):.4f}")
print()

# 2. Train RMSE
preds_cv = global_mean + user_bias_cv + np.array([row[1] for row in rows]) + Q @ user_vector_cv
rmse_cv  = np.sqrt(np.mean((preds_cv - np.array(actuals)) ** 2))
print(f"Train RMSE — lstsq: {rmse:.4f}  |  Ridge(0.1): {rmse_ridge:.4f}  |  RidgeCV: {rmse_cv:.4f}")
print()

# 3. Top-10 predicted movies across all matched items (training set only — full sweep comes later)
all_preds = []
for (qi, bi, r, title) in rows:
    pred = global_mean + user_bias_cv + bi + user_vector_cv @ qi
    all_preds.append({'title': title, 'actual': r, 'predicted': round(pred, 3)})

pd.DataFrame(all_preds).sort_values('predicted', ascending=False).head(10)

## Finer alpha search — RMSE vs prediction spread

LOOCV on 35 points is noisy: small RMSE differences between alphas aren't meaningful, but prediction spread (std dev across all movies) tells us whether the model still differentiates between films. We want the lowest alpha where LOOCV RMSE has levelled off — not the strict minimum.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

fine_alphas = [0.1, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
results_cv  = []

for a in fine_alphas:
    m = Ridge(alpha=a, fit_intercept=True)

    # LOOCV RMSE
    scores = cross_val_score(m, Q, b_target, cv=len(rows),
                             scoring='neg_mean_squared_error')
    loocv_rmse = np.sqrt(-scores.mean())

    # Fit on full data → measure prediction spread across training items
    m.fit(Q, b_target)
    preds_all = global_mean + m.intercept_ + np.array([row[1] for row in rows]) + Q @ m.coef_
    spread = preds_all.std()

    results_cv.append({'alpha': a, 'loocv_rmse': loocv_rmse, 'pred_spread': spread})

df_cv = pd.DataFrame(results_cv)
print(df_cv.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(df_cv['alpha'], df_cv['loocv_rmse'], 'b-o', label='LOOCV RMSE')
ax2.plot(df_cv['alpha'], df_cv['pred_spread'], 'r-s', label='Pred spread (std)')
ax1.set_xlabel('alpha'); ax1.set_ylabel('LOOCV RMSE', color='b')
ax2.set_ylabel('Prediction spread (std)', color='r')
ax1.set_title('Alpha: RMSE vs Spread trade-off')
fig.tight_layout(); plt.show()

## Preview — top predictions across all MovieLens items (alpha=3.0)

Fit the user vector at alpha=3.0, then sweep all ~27K item vectors to see what the model recommends before saving anything to disk.

In [ ]:
from sklearn.linear_model import Ridge

# Fit final user vector
ridge_final = Ridge(alpha=3.0, fit_intercept=True)
ridge_final.fit(Q, b_target)
user_bias_final   = ridge_final.intercept_
user_vector_final = ridge_final.coef_

# Load movie titles
movies_df = pd.read_csv(data_path / 'ml-32m' / 'movies.csv')  # movieId, title, genres
mid_to_title = dict(zip(movies_df['movieId'], movies_df['title']))

# Sweep all item vectors
all_scores = (global_mean + user_bias_final
              + item_biases
              + item_vectors @ user_vector_final)   # vectorised dot product

scores_df = pd.DataFrame({
    'movieId': movieids,
    'title':   [mid_to_title.get(mid, '?') for mid in movieids],
    'predicted': all_scores
})

# Exclude movies already rated
rated_ids = set(matched['movieId'].astype(int))
scores_df = scores_df[~scores_df['movieId'].isin(rated_ids)]

print(f"Scoring {len(scores_df):,} unrated movies\n")
scores_df.sort_values('predicted', ascending=False).head(25)

In [ ]:
# Look up specific titles (searches across ALL movies, including already-rated ones)
# Mean rating per movie from the full MovieLens dataset
ml_ratings = pd.read_csv(data_path / 'ml-32m' / 'ratings.csv', usecols=['movieId', 'rating'])
movie_stats = ml_ratings.groupby('movieId')['rating'].agg(mean_rating='mean', num_ratings='count').reset_index()

all_scores_df = pd.DataFrame({
    'movieId':   movieids,
    'title':     [mid_to_title.get(mid, '?') for mid in movieids],
    'predicted': all_scores,
    'rated':     [mid in rated_ids for mid in movieids]
}).merge(movie_stats, on='movieId', how='left')

all_scores_df['mean_rating'] = all_scores_df['mean_rating'].round(3)

search_terms = ['borat']
mask = all_scores_df['title'].str.lower().str.contains('|'.join(search_terms))
all_scores_df[mask].sort_values('predicted', ascending=False)[
    ['title', 'predicted', 'mean_rating', 'num_ratings', 'rated']
]

,title,predicted,mean_rating,num_ratings,rated
3166,Avengers: Infinity War - Part I (2018),4.028913,3.897,14645,False
370,Inception (2010),4.015872,4.157,57931,False
1369,Avengers: Infinity War - Part II (2019),3.981506,3.871,11811,False
866,"Avengers, The (2012)",3.850565,3.734,26315,False
3905,Avengers: Age of Ultron (2015),3.636182,3.487,12099,False
20633,Crippled Avengers (Can que) (Return of the 5 D...,3.609929,3.339,31,False
23203,The New Adventures of the Elusive Avengers (1968),3.502171,3.562,24,False
9301,Ultimate Avengers 2 (2006),3.387100,3.202,52,False
1812,Something Wicked This Way Comes (1983),3.385794,3.391,1481,False
5983,"Extremely Wicked, Shockingly Evil and Vile (2019)",3.364408,3.295,430,False


## Decompose predictions — how much is personalization vs popularity?

`predicted = global_mean + b_i + b_u + u·q_i`

If the `u·q_i` term has a tiny range relative to `b_i`, predictions will just track average movie ratings rather than personal taste.

In [ ]:
personalization = item_vectors @ user_vector_final   # u·q_i for every movie
popularity      = item_biases                        # b_i for every movie

print("=== Component ranges across all ~27K movies ===")
print(f"global_mean          : {global_mean:.4f}  (constant)")
print(f"b_i  (popularity)    : [{popularity.min():.4f}, {popularity.max():.4f}]  std={popularity.std():.4f}")
print(f"u·q_i (personal)     : [{personalization.min():.4f}, {personalization.max():.4f}]  std={personalization.std():.4f}")
print(f"b_u  (user bias)     : {user_bias_final:.4f}  (constant shift)")
print()
print(f"Ratio personal/popularity std: {personalization.std() / popularity.std():.3f}x")

## Joint training — personal user vector from SVD

Instead of estimating the user vector post-hoc via Ridge (fold-in), the personal ratings were injected directly into the MovieLens training set and the user vector was learned jointly with all item vectors. Load it and compare against the fold-in approach.

In [ ]:
# Load jointly-trained user vector
jt = np.load(data_path / 'svd_user_vector.npz')
user_vector_jt = jt['user_vector']   # shape (k,)
user_bias_jt   = jt['user_bias'][0]  # scalar

print(f"Joint  — bias: {user_bias_jt:.4f}  |  norm: {np.linalg.norm(user_vector_jt):.4f}")
print(f"RidgeCV— bias: {user_bias_final:.4f}  |  norm: {np.linalg.norm(user_vector_final):.4f}")

# Component decomposition — joint training
personalization_jt = item_vectors @ user_vector_jt
popularity         = item_biases

print("\n=== Component ranges — Joint Training ===")
print(f"global_mean      : {global_mean:.4f}")
print(f"b_i (popularity) : [{popularity.min():.4f}, {popularity.max():.4f}]  std={popularity.std():.4f}")
print(f"u·q_i (personal) : [{personalization_jt.min():.4f}, {personalization_jt.max():.4f}]  std={personalization_jt.std():.4f}")
print(f"b_u (user bias)  : {user_bias_jt:.4f}")
print(f"\nRatio personal/popularity std: {personalization_jt.std() / popularity.std():.3f}x")
print(f"(vs RidgeCV fold-in:           {(item_vectors @ user_vector_final).std() / popularity.std():.3f}x)")

In [ ]:
# Top-25 predictions using jointly-trained vector (excluding already-rated films)
all_scores_jt = global_mean + user_bias_jt + item_biases + personalization_jt

scores_jt_df = pd.DataFrame({
    'movieId':   movieids,
    'title':     [mid_to_title.get(mid, '?') for mid in movieids],
    'predicted': all_scores_jt,
}).merge(movie_stats, on='movieId', how='left')

scores_jt_df = scores_jt_df[~scores_jt_df['movieId'].isin(rated_ids)]
scores_jt_df['mean_rating'] = scores_jt_df['mean_rating'].round(3)

scores_jt_df.sort_values('predicted', ascending=False).head(25)[
    ['title', 'predicted', 'mean_rating']
]